# GP-5: Model Compression

Age regression (UTKFace → ResNet18) — Knowledge Distillation, Quantization, Pruning, Caching.

**Stages:**
1. Setup & data loading
2. Train / load **teacher** (ResNet50)
3. Train **baseline student** (ResNet18, no teacher)
4. **Knowledge Distillation** (ResNet50 → ResNet18)
5. **Quantization** (dynamic + PTQ) on distilled model
6. **Pruning** + fine-tune on distilled model
7. **Caching** demo (embedding + prediction cache)
8. Final comparison tables

## 0. Setup

In [11]:
import sys, os
# Make sure repo root is on the path when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
# Set CWD to repo root so all relative paths (configs, models, data) resolve correctly
os.chdir(repo_root)

import copy
import time
import yaml
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.models.resent_model import create_resnet18, create_resnet50, get_device
from src.data.preprocessing import load_data, create_age_bins, train_val_split
from src.data.dataset import AgeDataset
from src.utils import rmse, mae, acc_at_k, compute_metrics_by_bin

from src.compression.training import get_transforms, train_two_stage, evaluate
from src.compression.distillation import train_distillation
from src.compression.quantization import (
    quantize_dynamic, prepare_backbone_for_ptq, calibrate, convert_prepared_quantized_model
)
from src.compression.pruning import prune_model, remove_pruning, compute_sparsity
from src.compression.caching import CachedAgePredictor, image_hash
from src.compression.benchmark import (
    measure_model_size_mb, measure_latency, measure_peak_memory_mb,
    evaluate_quality, compare_models
)

device = get_device()
print(f'Device: {device}')
print(f'Repo root: {repo_root}')

Device: mps
Repo root: /Users/baranczov/GitHub Projects/ML_HSE


In [12]:
with open('configs/compression.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Quick-run mode ────────────────────────────────────────────────────────
# Set QUICK = True to use fewer epochs and a data subset (for testing the
# pipeline without waiting hours). Set to False for a proper training run.
QUICK = True

if QUICK:
    cfg['training']['epochs_head']     = 1
    cfg['training']['epochs_finetune'] = 2
    cfg['training']['patience']        = 5  # don't stop early in quick mode
    cfg['pruning']['finetune_epochs']  = 1
    cfg['quantization']['calibration_samples'] = 50
    print('QUICK mode ON: few epochs, small calibration set')

os.makedirs(cfg['paths']['models_dir'],  exist_ok=True)
os.makedirs(cfg['paths']['results_dir'], exist_ok=True)

QUICK mode ON: few epochs, small calibration set


## 1. Data

In [13]:
torch.manual_seed(cfg['data']['seed'])
np.random.seed(cfg['data']['seed'])

df, bad = load_data(cfg['paths']['data_dir'])
print(f'Loaded {len(df)} images ({bad} failed to parse)')

df, bin_labels = create_age_bins(df)
train_df, val_df = train_val_split(
    df,
    test_size=cfg['data']['test_size'],
    random_state=cfg['data']['seed']
)
print(f'Train: {len(train_df)}  Val: {len(val_df)}')

Loaded 23703 images (5 failed to parse)
Train: 18962  Val: 4741


In [14]:
train_tfm, val_tfm = get_transforms(cfg['data']['img_size'])

def make_loaders(train_df, val_df, train_tfm, val_tfm, cfg):
    train_ds = AgeDataset(train_df, train_tfm)
    val_ds   = AgeDataset(val_df,   val_tfm)
    kw = dict(batch_size=cfg['data']['batch_size'],
              num_workers=cfg['data']['num_workers'],
              pin_memory=True)
    return (DataLoader(train_ds, shuffle=True,  **kw),
            DataLoader(val_ds,   shuffle=False, **kw))

train_loader, val_loader = make_loaders(train_df, val_df, train_tfm, val_tfm, cfg)

## 2. Teacher — ResNet50

In [15]:
teacher_ckpt = cfg['distillation']['teacher_ckpt']

teacher = create_resnet50(pretrained=True, dropout=cfg['model']['dropout'])
teacher = teacher.to(device)

if os.path.exists(teacher_ckpt):
    print(f'Loading teacher from {teacher_ckpt}')
    teacher.load_state_dict(torch.load(teacher_ckpt, map_location=device))
else:
    print('Training teacher ResNet50 ...')
    teacher = train_two_stage(teacher, train_loader, val_loader, cfg, device)
    torch.save(teacher.state_dict(), teacher_ckpt)
    print(f'Teacher saved → {teacher_ckpt}')

teacher_metrics, _, _ = evaluate(teacher, val_loader, device)
print(f'Teacher  RMSE={teacher_metrics["rmse"]:.4f}  MAE={teacher_metrics["mae"]:.4f}')

Training teacher ResNet50 ...
=== Stage 1: head only ===


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[head] epoch 1/1  loss=16.4396  val_rmse=21.5396  val_mae=15.5464

=== Stage 2: full fine-tune (starting from head RMSE 21.5396) ===


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[finetune] epoch 1/2  loss=6.0692  val_rmse=9.1154  val_mae=6.7067


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[finetune] epoch 2/2  loss=4.1244  val_rmse=8.0062  val_mae=5.5980

Done. Best val RMSE: 8.0062
Teacher saved → models/teacher_resnet50.pth


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Teacher  RMSE=8.0062  MAE=5.5980


## 3. Baseline Student — ResNet18 (no distillation)

In [16]:
baseline_ckpt = os.path.join(cfg['paths']['models_dir'], 'baseline_resnet18.pth')

student_baseline = create_resnet18(pretrained=True, dropout=cfg['model']['dropout'])
student_baseline = student_baseline.to(device)

if os.path.exists(baseline_ckpt):
    print(f'Loading baseline student from {baseline_ckpt}')
    student_baseline.load_state_dict(torch.load(baseline_ckpt, map_location=device))
else:
    print('Training baseline ResNet18 (no teacher) ...')
    student_baseline = train_two_stage(student_baseline, train_loader, val_loader, cfg, device)
    torch.save(student_baseline.state_dict(), baseline_ckpt)
    print(f'Baseline saved → {baseline_ckpt}')

baseline_metrics, _, _ = evaluate(student_baseline, val_loader, device)
print(f'Baseline RMSE={baseline_metrics["rmse"]:.4f}  MAE={baseline_metrics["mae"]:.4f}')

Training baseline ResNet18 (no teacher) ...
=== Stage 1: head only ===


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[head] epoch 1/1  loss=14.0440  val_rmse=19.0184  val_mae=13.9182

=== Stage 2: full fine-tune (starting from head RMSE 19.0184) ===


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[finetune] epoch 1/2  loss=5.1608  val_rmse=8.2026  val_mae=5.8068


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[finetune] epoch 2/2  loss=3.9989  val_rmse=7.8355  val_mae=5.6788

Done. Best val RMSE: 7.8355
Baseline saved → models/baseline_resnet18.pth


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Baseline RMSE=7.8355  MAE=5.6788


## 4. Knowledge Distillation — ResNet50 → ResNet18

In [17]:
distilled_ckpt = os.path.join(cfg['paths']['models_dir'], 'distilled_resnet18.pth')

student_distilled = create_resnet18(pretrained=True, dropout=cfg['model']['dropout'])
student_distilled = student_distilled.to(device)

if os.path.exists(distilled_ckpt):
    print(f'Loading distilled student from {distilled_ckpt}')
    student_distilled.load_state_dict(torch.load(distilled_ckpt, map_location=device))
else:
    print('Knowledge distillation: ResNet50 → ResNet18 ...')
    # Stage 1: train head of student first (warm-up)
    student_distilled = train_two_stage(student_distilled, train_loader, val_loader, cfg, device)
    # Stage 2: distill with teacher
    student_distilled = train_distillation(
        student_distilled, teacher, train_loader, val_loader, cfg, device
    )
    torch.save(student_distilled.state_dict(), distilled_ckpt)
    print(f'Distilled model saved → {distilled_ckpt}')

distilled_metrics, _, _ = evaluate(student_distilled, val_loader, device)
print(f'Distilled RMSE={distilled_metrics["rmse"]:.4f}  MAE={distilled_metrics["mae"]:.4f}')

Knowledge distillation: ResNet50 → ResNet18 ...
=== Stage 1: head only ===


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[head] epoch 1/1  loss=13.9574  val_rmse=18.8010  val_mae=13.8610

=== Stage 2: full fine-tune (starting from head RMSE 18.8010) ===


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[finetune] epoch 1/2  loss=5.2216  val_rmse=8.4025  val_mae=6.0114


...

[distill] epoch 1/2  loss=29.5005  task=4.0206  kd=25.4799  val_rmse=8.1374  val_mae=5.8455


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[distill] epoch 2/2  loss=23.2984  task=3.8049  kd=19.4935  val_rmse=8.0090  val_mae=5.6931
Distillation done. Best val RMSE: 8.0090
Distilled model saved → models/distilled_resnet18.pth


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Distilled RMSE=8.0090  MAE=5.6931


### Distillation results — comparison

Analogous to Table 1 from the reference report.

In [18]:
import pandas as pd

distill_rows = []
for name, m in [('Teacher (ResNet50)', teacher),
                ('Student w/o teacher (ResNet18)', student_baseline),
                ('Distilled student (ResNet18)', student_distilled)]:
    metrics, t, p = evaluate(m, val_loader, device)
    distill_rows.append({
        'Model': name,
        'RMSE':    round(metrics['rmse'], 4),
        'MAE':     round(metrics['mae'],  4),
        'Acc@5':   round(acc_at_k(t, p, k=5),  4),
        'Acc@10':  round(acc_at_k(t, p, k=10), 4),
        'Size MB': round(measure_model_size_mb(m), 2),
    })

df_distill = pd.DataFrame(distill_rows).set_index('Model')
print('\nTable 1 — Distillation results')
print(df_distill.to_string())

/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 


Table 1 — Distillation results
                                  RMSE     MAE   Acc@5  Acc@10  Size MB
Model                                                                  
Teacher (ResNet50)              8.0062  5.5980  0.5950  0.8325    89.98
Student w/o teacher (ResNet18)  7.8355  5.6788  0.5691  0.8254    42.71
Distilled student (ResNet18)    8.0090  5.6931  0.5845  0.8209    42.71


## 5. Quantization (applied to distilled model)

In [19]:
example_batch = torch.randn(1, 3, cfg['data']['img_size'], cfg['data']['img_size'])
cpu_device = torch.device('cpu')

# 5a. Dynamic quantization
print('Applying dynamic quantization ...')
model_dynamic = quantize_dynamic(student_distilled)
dyn_metrics = evaluate_quality(model_dynamic, val_loader, device=cpu_device)
print(f'Dynamic quant  RMSE={dyn_metrics["rmse"]:.4f}  MAE={dyn_metrics["mae"]:.4f}')

# 5b. PTQ
# Calibrate on train_loader (not val_loader) to avoid leaking eval data into
# activation-range statistics, which would produce optimistic PTQ quality numbers.
print('\nApplying PTQ ...')
prepared = prepare_backbone_for_ptq(
    student_distilled,
    example_inputs=(example_batch,),
    backend=cfg['quantization']['backend']
)
prepared = calibrate(prepared, train_loader,
                     calibration_samples=cfg['quantization']['calibration_samples'])
model_ptq = convert_prepared_quantized_model(prepared)
ptq_metrics = evaluate_quality(model_ptq, val_loader, device=cpu_device)
print(f'PTQ            RMSE={ptq_metrics["rmse"]:.4f}  MAE={ptq_metrics["mae"]:.4f}')

Applying dynamic quantization ...


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[W qlinear_dynamic.cpp:247] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


Dynamic quant  RMSE=8.0127  MAE=5.6963

Applying PTQ ...


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

PTQ            RMSE=8.1147  MAE=5.7711


In [21]:
# Latency benchmark (CPU, single image)
example_single = torch.randn(1, 3, cfg['data']['img_size'], cfg['data']['img_size'])

def bench(m, label):
    stats = measure_latency(m, example_single, device=cpu_device, n_warmup=20, n_iter=200)
    size  = measure_model_size_mb(m)
    print(f'{label:40s}  size={size:.1f} MB  '
          f'latency={stats["latency_ms"]:.2f} ms  '
          f'throughput={stats["throughput_img_per_s"]:.1f} img/s')

bench(student_distilled.cpu(), 'Distilled float32 (CPU)')
bench(model_dynamic,           'Dynamic quantized  (CPU)')
bench(model_ptq,               'PTQ int8           (CPU)')

Distilled float32 (CPU)                   size=42.7 MB  latency=7.00 ms  throughput=142.8 img/s
Dynamic quantized  (CPU)                  size=42.7 MB  latency=7.24 ms  throughput=138.2 img/s
PTQ int8           (CPU)                  size=10.7 MB  latency=9.81 ms  throughput=101.9 img/s


## 6. Pruning (applied to distilled model)

In [22]:
pruned_ckpt = os.path.join(cfg['paths']['models_dir'], 'pruned_resnet18.pth')

model_pruned = copy.deepcopy(student_distilled).to(device)

print(f'Sparsity before pruning: {compute_sparsity(model_pruned):.1%}')
prune_model(model_pruned, amount=cfg['pruning']['amount'],
            structured=cfg['pruning']['structured'])
print(f'Sparsity after  pruning: {compute_sparsity(model_pruned):.1%}')

# Evaluate before fine-tune to show quality drop
pre_ft_metrics, _, _ = evaluate(model_pruned, val_loader, device)
print(f'RMSE before fine-tune: {pre_ft_metrics["rmse"]:.4f}')

# Fine-tune
ft_cfg = copy.deepcopy(cfg)
ft_cfg['training']['epochs_head']     = 0  # skip head-only stage
ft_cfg['training']['epochs_finetune'] = cfg['pruning']['finetune_epochs']
ft_cfg['training']['patience']        = cfg['pruning']['finetune_epochs'] + 1

remove_pruning(model_pruned)  # bake masks before fine-tuning

print('Fine-tuning pruned model ...')
model_pruned = train_two_stage(model_pruned, train_loader, val_loader, ft_cfg, device)

post_ft_metrics, _, _ = evaluate(model_pruned, val_loader, device)
print(f'RMSE after  fine-tune: {post_ft_metrics["rmse"]:.4f}')

torch.save(model_pruned.state_dict(), pruned_ckpt)
print(f'Pruned model saved → {pruned_ckpt}')

Sparsity before pruning: 0.0%
Sparsity after  pruning: 50.0%


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


RMSE before fine-tune: 7.9246
Fine-tuning pruned model ...
=== Stage 1: head only ===

=== Stage 2: full fine-tune (starting from head RMSE inf) ===


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

[finetune] epoch 1/1  loss=3.6582  val_rmse=7.8539  val_mae=5.6556

Done. Best val RMSE: 7.8539


/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


RMSE after  fine-tune: 7.8539
Pruned model saved → models/pruned_resnet18.pth


## 7. Caching demo

In [23]:
from PIL import Image as PILImage

sample_paths = val_df['path'].iloc[:50].tolist()
images = [PILImage.open(p).convert('RGB') for p in sample_paths]

predictor = CachedAgePredictor(student_distilled.cpu(), device=cpu_device,
                               max_cache_size=1000)

predictor.reset_stats()
t0 = time.perf_counter()
ages_pass1 = predictor.predict_batch(images, val_tfm)
t_cold = time.perf_counter() - t0

predictor.reset_stats()
t0 = time.perf_counter()
ages_pass2 = predictor.predict_batch(images, val_tfm)
t_warm = time.perf_counter() - t0

print(f'Cold pass: {t_cold*1000:.1f} ms  ({t_cold/len(images)*1000:.2f} ms/img)')
print(f'Warm pass: {t_warm*1000:.1f} ms  ({t_warm/len(images)*1000:.2f} ms/img)')
print(f'Speedup: {t_cold/t_warm:.1f}x')
print()
predictor.print_stats()

predictor.pred_cache.clear()
predictor.reset_stats()
t0 = time.perf_counter()
_ = predictor.predict_batch(images, val_tfm)  
t_emb = time.perf_counter() - t0
print(f'\nEmbedding-cache pass: {t_emb*1000:.1f} ms')
predictor.print_stats()

Cold pass: 1002.3 ms  (20.05 ms/img)
Warm pass: 619.8 ms  (12.40 ms/img)
Speedup: 1.6x

PredictionCache: hits=50, misses=0, hit_rate=100.0%
EmbeddingCache:  hits=0, misses=0, hit_rate=0.0%

Embedding-cache pass: 632.8 ms
PredictionCache: hits=0, misses=50, hit_rate=0.0%
EmbeddingCache:  hits=50, misses=0, hit_rate=100.0%


## 8. Final comparison tables

Analogous to Tables 1–2 in the reference report.

In [25]:
# Table 1: quality comparison
quality_models = {
    'Teacher (ResNet50)':            teacher,
    'Student w/o teacher':           student_baseline,
    'Distilled student':             student_distilled,
    'Pruned + fine-tuned':           model_pruned,
}

rows = []
for name, m in quality_models.items():
    m.to(device)  # restore to compute device in case a bench cell moved it to CPU
    metrics, t, p = evaluate(m, val_loader, device)
    rows.append({
        'Model':    name,
        'RMSE':     round(metrics['rmse'], 4),
        'MAE':      round(metrics['mae'],  4),
        'Acc@5':    round(acc_at_k(t, p, k=5), 4),
        'Acc@10':   round(acc_at_k(t, p, k=10), 4),
        'Size MB':  round(measure_model_size_mb(m), 2),
    })

df_quality = pd.DataFrame(rows).set_index('Model')
print('Table 1 — Quality vs. compression')
print(df_quality.to_string())

/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only 

Table 1 — Quality vs. compression
                       RMSE     MAE   Acc@5  Acc@10  Size MB
Model                                                       
Teacher (ResNet50)   8.0062  5.5980  0.5950  0.8325    89.98
Student w/o teacher  7.8355  5.6788  0.5691  0.8254    42.71
Distilled student    8.0090  5.6931  0.5845  0.8209    42.71
Pruned + fine-tuned  7.8539  5.6556  0.5765  0.8298    42.71


In [26]:
deploy_models = {
    'Distilled float32': student_distilled.cpu(),
    'Dynamic quant':     model_dynamic,
    'PTQ int8':          model_ptq,
    'Pruned float32':    model_pruned.cpu(),
}

deploy_rows = []
for name, m in deploy_models.items():
    size  = measure_model_size_mb(m)
    stats = measure_latency(m, example_single, device=cpu_device, n_warmup=20, n_iter=200)
    mem   = measure_peak_memory_mb(m, example_single, device=cpu_device)
    deploy_rows.append({
        'Model':             name,
        'Size MB':           round(size, 2),
        'Latency ms':        round(stats['latency_ms'], 2),
        'Throughput img/s':  round(stats['throughput_img_per_s'], 1),
        'Peak RAM MB':       round(mem, 2),
    })

df_deploy = pd.DataFrame(deploy_rows).set_index('Model')
print('Table 2 — Deployment metrics (CPU inference)')
print(df_deploy.to_string())

Table 2 — Deployment metrics (CPU inference)
                   Size MB  Latency ms  Throughput img/s  Peak RAM MB
Model                                                                
Distilled float32    42.71        7.13             140.3         0.05
Dynamic quant        42.71        7.15             139.8         0.00
PTQ int8             10.70        9.76             102.4         0.00
Pruned float32       42.71        6.85             145.9         0.00


In [27]:
df_quality.to_csv(os.path.join(cfg['paths']['results_dir'], 'table1_quality.csv'))
df_deploy.to_csv(os.path.join(cfg['paths']['results_dir'],  'table2_deployment.csv'))
print('Tables saved to', cfg['paths']['results_dir'])

Tables saved to results/


## Per-age-bin RMSE (distilled model)

Shows which age groups are most affected by compression.

In [28]:
student_distilled.to(device)
_, trues, preds = evaluate(student_distilled, val_loader, device)

val_res = val_df.reset_index(drop=True).copy()
val_res['pred_age'] = preds

by_bin = compute_metrics_by_bin(
    val_res, 'age', 'pred_age',
    bins=[0,3,13,20,30,40,50,60,70,80,117],
    labels=['0-2','3-12','13-19','20-29','30-39','40-49','50-59','60-69','70-79','80+']
)
print('RMSE by age bin (distilled student):')
print(by_bin[['n','rmse','mae']].to_string())

/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/baranczov/GitHub Projects/ML_HSE/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


RMSE by age bin (distilled student):
            n       rmse        mae
age_bin                            
0-2       321   4.202202   1.803194
3-12      362   7.029526   4.059156
13-19     236   6.601168   5.575155
20-29    1469   4.737241   3.404369
30-39     907   6.658508   5.574314
40-49     449   9.370098   7.924815
50-59     460  11.927947   9.604986
60-69     263  12.826457   9.952630
70-79     140  13.244132  10.717147
80+       134  13.492947  11.012027
